In [99]:
!pip install imblearn

In [100]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report,confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, Activation
from tensorflow.keras.utils import to_categorical
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

import seaborn as sns

In [101]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, Activation
from tensorflow.keras.utils import to_categorical

# Load dataset
data = pd.read_csv('/content/NF-BoT-IoT.csv')




In [102]:
data.columns

Index(['IPV4_SRC_ADDR', 'L4_SRC_PORT', 'IPV4_DST_ADDR', 'L4_DST_PORT',
       'PROTOCOL', 'L7_PROTO', 'IN_BYTES', 'OUT_BYTES', 'IN_PKTS', 'OUT_PKTS',
       'TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS', 'Label', 'Attack'],
      dtype='object')

In [103]:
#data.rename(columns={'Label_encoded': 'Label'}, inplace=True)

In [104]:
data.drop(['IPV4_SRC_ADDR','IPV4_DST_ADDR', 'Attack' ],axis=1,inplace=True)

In [105]:
data.describe()

,L4_SRC_PORT,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,OUT_BYTES,IN_PKTS,OUT_PKTS,TCP_FLAGS,FLOW_DURATION_MILLISECONDS,Label
count,600100.000000,600100.000000,600100.000000,600100.000000,6.001000e+05,6.001000e+05,600100.000000,600100.000000,600100.000000,6.001000e+05,600100.000000
mean,46528.901980,7949.493863,6.584854,8.956318,9.222368e+03,6.997822e+03,12.433891,5.696411,21.856519,3.468594e+06,0.976906
std,12036.388811,14101.151534,2.567061,34.969431,4.734768e+05,8.214194e+05,246.004187,191.421366,8.114914,1.665532e+06,0.150204
min,0.000000,0.000000,1.000000,0.000000,2.800000e+01,0.000000e+00,1.000000,0.000000,0.000000,0.000000e+00,0.000000
25%,39180.000000,80.000000,6.000000,0.000000,4.400000e+01,4.000000e+01,1.000000,1.000000,22.000000,4.277976e+06,1.000000
50%,47872.500000,1864.000000,6.000000,0.000000,4.400000e+01,4.000000e+01,1.000000,1.000000,22.000000,4.294966e+06,1.000000
75%,55307.000000,8009.000000,6.000000,7.000000,1.120000e+02,4.000000e+01,2.000000,1.000000,22.000000,4.294967e+06,1.000000
max,65535.000000,65535.000000,17.000000,244.000000,2.282235e+08,2.432197e+08,37817.000000,61110.000000,214.000000,4.294967e+06,1.000000


In [106]:
data

,L4_SRC_PORT,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,OUT_BYTES,IN_PKTS,OUT_PKTS,TCP_FLAGS,FLOW_DURATION_MILLISECONDS,Label
0,52670,53,17,5.212,71,126,1,1,0,4294966,0
1,49160,4444,6,0.000,217753000,199100,4521,4049,24,4176249,1
2,3456,80,17,0.000,8508021,8918372,9086,9086,0,4175916,0
3,80,8080,6,7.000,8442138,9013406,9086,9086,0,4175916,0
4,80,80,6,7.000,8374706,0,9086,0,0,4175916,0
...,...,...,...,...,...,...,...,...,...,...,...
600095,80,80,6,7.000,2330065,0,2523,0,0,4263037,0
600096,0,0,6,0.000,1054423,0,1513,0,0,4263062,0
600097,365,565,17,0.000,62422,0,1357,0,0,4263062,0
600098,50850,8883,6,222.178,11300,1664,32,32,24,4264935,0


In [124]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import KMeansSMOTE
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans  # Import KMeans for custom estimator

# Function to split data into training and test sets
def simple_split(data):
    train_data, test_data = train_test_split(data, test_size=0.3, random_state=42)
    return train_data, test_data

# Splitting the dataset
train_data, test_data = simple_split(data)

# Separating features (X) and target (y)
X_train = train_data.drop(columns=['Label'])
y_train = train_data['Label']

# Standardizing features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Applying K-Means SMOTE to handle class imbalance
# Use kmeans_estimator to specify the desired number of clusters
kmeans_smote = KMeansSMOTE(random_state=42, k_neighbors=2,
                           cluster_balance_threshold=0.1,
                           kmeans_estimator=KMeans(n_clusters=10))  # Specify n_clusters here
X_resampled, y_resampled = kmeans_smote.fit_resample(X_train_scaled, y_train)

# Now `X_resampled` and `y_resampled` contain the oversampled data
# Convert resampled data back to a DataFrame
X_resampled_df = pd.DataFrame(X_resampled, columns=X_train.columns)
y_resampled_df = pd.DataFrame(y_resampled, columns=['Label'])

# Combine features and target into a single DataFrame
train_data = pd.concat([X_resampled_df, y_resampled_df], axis=1)

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/utils/_tags.py:354: FutureWarning: The KMeansSMOTE or classes from which it inherits use `_get_tags` and `_more_tags`. Please define the `__sklearn_tags__` method, or inherit from `sklearn.base.BaseEstimator` and/or other appropriate mixins such as `sklearn.base.TransformerMixin`, `sklearn.base.ClassifierMixin`, `sklearn.base.RegressorMixin`, and `sklearn.base.OutlierMixin`. From scikit-learn 1.7, not defining `__sklearn_tags__` will raise an error.
  warnings.warn(


In [125]:
train_data

,L4_SRC_PORT,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,OUT_BYTES,IN_PKTS,OUT_PKTS,TCP_FLAGS,FLOW_DURATION_MILLISECONDS,Label
0,-0.315619,-0.563446,-0.227509,-0.256224,-0.017292,-0.008783,-0.046274,-0.030192,-2.423678,-2.082001,1
1,0.355574,-0.558410,-0.227509,-0.056145,-0.016025,-0.007658,-0.021985,-0.003307,1.114074,-2.082001,1
2,-0.315619,-0.409523,-0.227509,-0.256224,-0.017292,-0.008724,-0.046274,-0.024815,0.016151,0.496108,1
3,-0.473244,-0.515354,-0.227509,-0.256224,-0.017292,-0.008724,-0.046274,-0.024815,0.016151,0.496108,1
4,-0.815733,-0.558410,-0.227509,-0.056145,-0.015541,-0.007658,-0.021985,-0.003307,1.114074,-2.082001,1
...,...,...,...,...,...,...,...,...,...,...,...
820623,-3.859106,-0.558410,-0.227509,-0.056145,15.337380,-0.008783,36.645254,-0.030192,-2.667661,0.424136,0
820624,-3.835437,-0.524008,4.062162,-0.256224,0.418505,-0.008783,20.825741,-0.030192,-2.667661,0.424086,0
820625,-3.865750,-0.564085,-0.227509,-0.256224,7.355799,-0.008783,23.262699,-0.030192,-2.667661,0.424081,0
820626,-3.835437,-0.524008,4.062162,-0.256224,0.418505,-0.008783,20.825741,-0.030192,-2.667661,0.424084,0


In [126]:


# Remove rows that contain NaN values
train_data = train_data.dropna()

# Display the cleaned DataFrame
print(train_data)


        L4_SRC_PORT  L4_DST_PORT  PROTOCOL  L7_PROTO   IN_BYTES  OUT_BYTES  \
0         -0.315619    -0.563446 -0.227509 -0.256224  -0.017292  -0.008783   
1          0.355574    -0.558410 -0.227509 -0.056145  -0.016025  -0.007658   
2         -0.315619    -0.409523 -0.227509 -0.256224  -0.017292  -0.008724   
3         -0.473244    -0.515354 -0.227509 -0.256224  -0.017292  -0.008724   
4         -0.815733    -0.558410 -0.227509 -0.056145  -0.015541  -0.007658   
...             ...          ...       ...       ...        ...        ...   
820623    -3.859106    -0.558410 -0.227509 -0.056145  15.337380  -0.008783   
820624    -3.835437    -0.524008  4.062162 -0.256224   0.418505  -0.008783   
820625    -3.865750    -0.564085 -0.227509 -0.256224   7.355799  -0.008783   
820626    -3.835437    -0.524008  4.062162 -0.256224   0.418505  -0.008783   
820627    -3.194723    -0.558410 -0.227509 -0.056145  17.474593  13.025327   

          IN_PKTS   OUT_PKTS  TCP_FLAGS  FLOW_DURATION_MILLISEC

In [127]:
# Remove rows that contain NaN values
test_data = test_data.dropna()

# Display the cleaned DataFrame
print(test_data)

        L4_SRC_PORT  L4_DST_PORT  PROTOCOL  L7_PROTO  IN_BYTES  OUT_BYTES  \
74061         36256         6901         6       0.0        44         40   
201306        40281          163         6       0.0        44         40   
436503        50133         9898         6       0.0        44         40   
115099            0            0         1       0.0       168          0   
203771        45879         1216         6       0.0        44         40   
...             ...          ...       ...       ...       ...        ...   
159290        39747         5002         6       0.0        44         40   
569329        35070           80         6       7.0       770        770   
149250        55151         9103         6       0.0        44         40   
131589        37862         9000         6       0.0        44         40   
494274        58116         5718         6       0.0        60         40   

        IN_PKTS  OUT_PKTS  TCP_FLAGS  FLOW_DURATION_MILLISECONDS  Label  
7

In [128]:
train_data['Label'].value_counts()

,count
Label,
0,410315
1,410313


Echo State Network

In [129]:

# Manual Echo State Network Implementation
class ManualESN:
    def __init__(self, input_dim, reservoir_size, spectral_radius=0.95, sparsity=0.1):
        self.input_dim = input_dim
        self.reservoir_size = reservoir_size
        self.spectral_radius = spectral_radius
        self.sparsity = sparsity
        self.W_in = np.random.uniform(-1, 1, (reservoir_size, input_dim))
        self.W = np.random.uniform(-1, 1, (reservoir_size, reservoir_size))
        self.W[np.random.rand(*self.W.shape) > sparsity] = 0
        max_eig = max(abs(np.linalg.eigvals(self.W)))
        self.W *= spectral_radius / max_eig
        self.state = np.zeros(reservoir_size)

    def update(self, input_data):
        # Convert input_data to a NumPy array with float64 dtype
        input_data = np.array(input_data, dtype=np.float64)
        self.state = np.tanh(np.dot(self.W_in, input_data) + np.dot(self.W, self.state))
        return self.state

    def fit_transform(self, X):
        states = []
        for x in X:
            states.append(self.update(x))
        return np.array(states)


# DNN Training
class ESN_DNN:
    def __init__(self, reservoir_size=200, spectral_radius=0.95):
        self.esn = ManualESN(input_dim=train_data.drop('Label', axis=1).shape[1],
                             reservoir_size=reservoir_size, spectral_radius=spectral_radius)
        self.dnn = None

    def train(self, X, y):
        esn_output = self.esn.fit_transform(X)
        self.dnn = Sequential([
            Dense(64, activation='relu', input_shape=(esn_output.shape[1],)),
            Dropout(0.15),
            Dense(64, activation='relu'),
            Dropout(0.15),
            Dense(128, activation='relu'),
            Dropout(0.25),
            Dense(32, activation='relu'),
            Dense(len(np.unique(y)), activation='softmax')
        ])
        self.dnn.compile(optimizer='rmsprop', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        self.dnn.fit(esn_output, y, epochs=15, batch_size=64, verbose=1)

    def predict(self, X):
        esn_output = self.esn.fit_transform(X)
        return self.dnn.predict(esn_output)

# Train and evaluate the model
esn_dnn_model = ESN_DNN(reservoir_size=250, spectral_radius=0.95)
esn_dnn_model.train(train_data.drop('Label', axis=1).values, train_data['Label'].values)




/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/15
12823/12823 ━━━━━━━━━━━━━━━━━━━━ 55s 4ms/step - accuracy: 0.9902 - loss: 0.0370
Epoch 2/15
12823/12823 ━━━━━━━━━━━━━━━━━━━━ 78s 4ms/step - accuracy: 0.9923 - loss: 0.0266
Epoch 3/15
12823/12823 ━━━━━━━━━━━━━━━━━━━━ 82s 4ms/step - accuracy: 0.9928 - loss: 0.0252
Epoch 4/15
12823/12823 ━━━━━━━━━━━━━━━━━━━━ 84s 4ms/step - accuracy: 0.9928 - loss: 0.0249
Epoch 5/15
12823/12823 ━━━━━━━━━━━━━━━━━━━━ 51s 4ms/step - accuracy: 0.9929 - loss: 0.0252
Epoch 6/15
12823/12823 ━━━━━━━━━━━━━━━━━━━━ 50s 4ms/step - accuracy: 0.9930 - loss: 0.0246
Epoch 7/15
12823/12823 ━━━━━━━━━━━━━━━━━━━━ 51s 4ms/step - accuracy: 0.9930 - loss: 0.0254
Epoch 8/15
12823/12823 ━━━━━━━━━━━━━━━━━━━━ 82s 4ms/step - accuracy: 0.9930 - loss: 0.0245
Epoch 9/15
12823/12823 ━━━━━━━━━━━━━━━━━━━━ 51s 4ms/step - accuracy: 0.9930 - loss: 0.0253
Epoch 10/15
12823/12823 ━━━━━━━━━━━━━━━━━━━━ 81s 4ms/step - accuracy: 0.9930 - loss: 0.0254
Epoch 11/15
12823/12823 ━━━━━━━━━━━━━━━━━━━━ 50s 4ms/step - accuracy: 0.9931 - loss: 0.02

In [132]:
# Separating features (X) and target (y)
X_test = test_data.drop(columns=['Label'])
y_test = test_data['Label']

# Standardizing features
scaler1 = StandardScaler()
X_test_scaled = scaler1.fit_transform(X_test)

# Applying K-Means SMOTE to handle class imbalance
# Use kmeans_estimator to specify the desired number of clusters
kmeans_smote1 = KMeansSMOTE(random_state=42, k_neighbors=2,
                           cluster_balance_threshold=0.1,
                           kmeans_estimator=KMeans(n_clusters=10))  # Specify n_clusters here
X_resampled, y_resampled = kmeans_smote1.fit_resample(X_test_scaled, y_test)

# Now `X_resampled` and `y_resampled` contain the oversampled data
# Convert resampled data back to a DataFrame
X_resampled_df = pd.DataFrame(X_resampled, columns=X_test.columns)
y_resampled_df = pd.DataFrame(y_resampled, columns=['Label'])

# Combine features and target into a single DataFrame
test_data = pd.concat([X_resampled_df, y_resampled_df], axis=1)

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/utils/_tags.py:354: FutureWarning: The KMeansSMOTE or classes from which it inherits use `_get_tags` and `_more_tags`. Please define the `__sklearn_tags__` method, or inherit from `sklearn.base.BaseEstimator` and/or other appropriate mixins such as `sklearn.base.TransformerMixin`, `sklearn.base.ClassifierMixin`, `sklearn.base.RegressorMixin`, and `sklearn.base.OutlierMixin`. From scikit-learn 1.7, not defining `__sklearn_tags__` will raise an error.
  warnings.warn(


In [133]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Prediction for Test Data
y_test_true = test_data['Label'].values
y_test_pred_prob = esn_dnn_model.predict(test_data.drop('Label', axis=1).values)
y_test_pred = np.argmax(y_test_pred_prob, axis=1)

# Prediction for Train Data
y_train_true = train_data['Label'].values
y_train_pred_prob = esn_dnn_model.predict(train_data.drop('Label', axis=1).values)
y_train_pred = np.argmax(y_train_pred_prob, axis=1)

10996/10996 ━━━━━━━━━━━━━━━━━━━━ 20s 2ms/step
25645/25645 ━━━━━━━━━━━━━━━━━━━━ 49s 2ms/step


In [134]:


def evaluate_metrics(y_true, y_pred, average='macro'):
    metrics = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average=average, zero_division=0),
        "Recall": recall_score(y_true, y_pred, average=average),
        "F1 Score": f1_score(y_true, y_pred, average=average),


    }
    return metrics


# Evaluate for Test Data
test_metrics = evaluate_metrics(y_test_true, y_test_pred)
print("Test Data Metrics:")
for key, value in test_metrics.items():
    print(f"{key}: {value:.4f}")

# Evaluate for Train Data
train_metrics = evaluate_metrics(y_train_true, y_train_pred)
print("\nTrain Data Metrics:")
for key, value in train_metrics.items():
    print(f"{key}: {value:.4f}")

# Classification Report
print("\nClassification Report for Test Data:\n")
print(classification_report(y_test_true, y_test_pred))






Test Data Metrics:
Accuracy: 0.9925
Precision: 0.9925
Recall: 0.9925
F1 Score: 0.9925

Train Data Metrics:
Accuracy: 0.9934
Precision: 0.9934
Recall: 0.9934
F1 Score: 0.9934

Classification Report for Test Data:

              precision    recall  f1-score   support

           0       1.00      0.99      0.99    175931
           1       0.99      1.00      0.99    175928

    accuracy                           0.99    351859
   macro avg       0.99      0.99      0.99    351859
weighted avg       0.99      0.99      0.99    351859



In [115]:
y_test_pred_prob

array([[0.03751672, 0.9624833 ],
       [0.03751672, 0.9624833 ],
       [0.03751672, 0.9624833 ],
       ...,
       [0.03751672, 0.9624833 ],
       [0.03751672, 0.9624833 ],
       [0.03751672, 0.9624833 ]], dtype=float32)

In [116]:
y_train_pred

array([1, 1, 1, ..., 1, 1, 1])

In [117]:
y_test_true

array([1, 1, 1, ..., 1, 1, 1])